In [1]:
!whoami
!echo 24BAD405 EX06b

nishanth\nishanth
24BAD405 EX06b


In [1]:
import numpy as np
import time

def create_random_matrix(n):
    return np.random.randint(-9, 10, size=(n, n))

def matrix_add(A, B):
    return np.add(A, B)

def matrix_subtract(A, B):
    return np.subtract(A, B)

def split_matrix(M):
    n = M.shape[0] // 2
    return (M[:n, :n], M[:n, n:],
            M[n:, :n], M[n:, n:])

def combine_matrix(C11, C12, C21, C22):
    n = C11.shape[0]
    C = np.zeros((2*n, 2*n), dtype=C11.dtype)
    C[:n, :n]   = C11
    C[:n, n:]   = C12
    C[n:, :n]   = C21
    C[n:, n:]   = C22
    return C

In [2]:
def matmul_naive(A, B):
    n = A.shape[0]
    C = np.zeros((n, n), dtype=A.dtype)
    for i in range(n):
        for j in range(n):
            for k in range(n):
                C[i,j] += A[i,k] * B[k,j]
    return C

In [3]:
scalar_multiplications = 0

def reset_counter():
    global scalar_multiplications
    scalar_multiplications = 0

def get_counter():
    global scalar_multiplications
    return scalar_multiplications

def matmul_strassen(A, B):
    global scalar_multiplications
    
    n = A.shape[0]
    
    if n == 1:
        scalar_multiplications += 1
        return A * B
    
    A11, A12, A21, A22 = split_matrix(A)
    B11, B12, B21, B22 = split_matrix(B)
    
    M1 = matmul_strassen(A11 + A22, B11 + B22)
    M2 = matmul_strassen(A21 + A22, B11)
    M3 = matmul_strassen(A11, B12 - B22)
    M4 = matmul_strassen(A22, B21 - B11)
    M5 = matmul_strassen(A11 + A12, B22)
    M6 = matmul_strassen(A21 - A11, B11 + B12)
    M7 = matmul_strassen(A12 - A22, B21 + B22)
    
    C11 = M1 + M4 - M5 + M7
    C12 = M3 + M5
    C21 = M2 + M4
    C22 = M1 - M2 + M3 + M6
    
    return combine_matrix(C11, C12, C21, C22)

In [4]:
reset_counter()
A4 = create_random_matrix(4)
B4 = create_random_matrix(4)

C_naive = matmul_naive(A4, B4)
C_strassen = matmul_strassen(A4, B4)

print("Naive  vs  Strassen  match →", np.allclose(C_naive, C_strassen))
print("Naive multiplications    :", 4**3)
print("Strassen multiplications :", get_counter())

Naive  vs  Strassen  match → True
Naive multiplications    : 64
Strassen multiplications : 49


In [5]:
reset_counter()
A8 = create_random_matrix(8)
B8 = create_random_matrix(8)

_ = matmul_naive(A8, B8)
naive_mult_8 = 8**3

reset_counter()
_ = matmul_strassen(A8, B8)
strassen_mult_8 = get_counter()

print(f"8×8 naive multiplications    : {naive_mult_8}")
print(f"8×8 Strassen multiplications : {strassen_mult_8}")
print(f"Ratio (naive/Strassen)       : {naive_mult_8 / strassen_mult_8:.2f}x")

8×8 naive multiplications    : 512
8×8 Strassen multiplications : 343
Ratio (naive/Strassen)       : 1.49x


In [6]:
def next_power_of_2(x):
    return 1 if x == 0 else 2 ** (x-1).bit_length()

def pad_matrix(M, new_size):
    n = M.shape[0]
    if n == new_size:
        return M
    padded = np.zeros((new_size, new_size), dtype=M.dtype)
    padded[:n, :n] = M
    return padded

def unpad_matrix(M, original_size):
    return M[:original_size, :original_size]

def matmul_strassen_any_size(A, B):
    n = A.shape[0]
    m = B.shape[1]
    if A.shape[1] != B.shape[0]:
        raise ValueError("Matrix dimensions incompatible")
    
    size = max(n, A.shape[1], m)
    new_size = next_power_of_2(size)
    
    Ap = pad_matrix(A, new_size)
    Bp = pad_matrix(B, new_size)
    
    reset_counter()
    Cp = matmul_strassen(Ap, Bp)
    
    return unpad_matrix(Cp, n), get_counter()

In [7]:
for size in [3, 5, 6, 7]:
    print(f"\n=== {size}×{size} ===")
    
    A = create_random_matrix(size)
    B = create_random_matrix(size)
    
    C_np = np.dot(A, B)
    
    C_str, mults = matmul_strassen_any_size(A, B)
    
    print(f"NumPy vs padded-Strassen match → {np.allclose(C_np, C_str, atol=1e-10)}")
    print(f"Strassen scalar multiplications: {mults}")


=== 3×3 ===
NumPy vs padded-Strassen match → True
Strassen scalar multiplications: 49

=== 5×5 ===
NumPy vs padded-Strassen match → True
Strassen scalar multiplications: 343

=== 6×6 ===
NumPy vs padded-Strassen match → True
Strassen scalar multiplications: 343

=== 7×7 ===
NumPy vs padded-Strassen match → True
Strassen scalar multiplications: 343
